In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from pyomo.opt import *

In [2]:
df= pd.read_excel('app19.xlsx')
df.set_index('carvao', inplace=True)
df

,enxofre,po_carvao,libras_vapor
carvao,,,
1,1100,1.7,24000
2,3500,3.2,36000
3,1300,2.4,28000


In [3]:
limite_enxofre = 2500
limite_po_carvao = 2.8

In [ ]:
model = pyo.ConcreteModel()

model.carvao = pyo.Set(initialize=  df.index)
model.x = pyo.Var(model.carvao, bounds=(0, 1), domain=pyo.NonNegativeReals)

def obj(model):
    return sum(model.x[c]*df['libras_vapor'][c] for c in model.carvao)
model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

def restricao_valor_total(model):
    return sum(model.x[c] for c in model.carvao) <= 1.0
model.restricao_valor_total = pyo.Constraint(rule=restricao_valor_total)

def restricao_enxofre(model):
    return sum(model.x[c]*df['enxofre'][c] for c in model.carvao) <= limite_enxofre
model.restricao_enxofre = pyo.Constraint(rule=restricao_enxofre)

def restricao_po_carvao(model):
    return sum(model.x[c]*df['po_carvao'][c] for c in model.carvao) <= limite_po_carvao
model.restricao_po_carvao = pyo.Constraint(rule=restricao_po_carvao)



In [18]:
model.pprint()

1 Set Declarations
    carvao : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {1, 2, 3}

1 Var Declarations
    x : Size=3, Index=carvao
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :  None :     1 : False :  True : NonNegativeReals
          2 :     0 :  None :     1 : False :  True : NonNegativeReals
          3 :     0 :  None :     1 : False :  True : NonNegativeReals

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 24000*x[1] + 36000*x[2] + 28000*x[3]

3 Constraint Declarations
    restricao_enxofre : Size=1, Index=None, Active=True
        Key  : Lower : Body                              : Upper  : Active
        None :  -Inf : 1100*x[1] + 3500*x[2] + 1300*x[3] : 2500.0 :   True
    restricao_po_carvao : Size=1, Index=None, Active=True
        Key  : Lower : Body       

In [20]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpids0o0ev.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpnupgkc1o.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpnupgkc1o.pyomo.lp
Objective sense      : Maximize
Variables            :       3  [Box: 3]
Objective nonzeros   :       3
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       9
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min   : 24000.00         Max   : 36

In [30]:
for m in model.x:
    print(f'Quantidade do carvão {m}: {model.x[m].value:.2f} ou {model.x[m].value*100:.2f} %')

print(f"Produção total de vapor: {model.obj()}")

Quantidade do carvão 1: 0.06 ou 5.80 %
Quantidade do carvão 2: 0.55 ou 55.07 %
Quantidade do carvão 3: 0.39 ou 39.13 %
Produção total de vapor: 32173.913043478264
